In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("flights").getOrCreate()

# Create flights dataframe
flights = spark.createDataFrame(
    [(1, "A1", "B1"), (2, "A2", "B2"), (3, "A3", "B3")],
    StructType(
        [
            StructField("flight_id", IntegerType(), True),
            StructField("origin_airport", StringType(), True),
            StructField("destination_airport", StringType(), True),
        ]
    ),
)

# Create airports dataframe
airports = spark.createDataFrame(
    [
        ("A1", "San Francisco"),
        ("B1", "Los Angeles"),
        ("A2", "New York"),
        ("B2", "Boston"),
        ("A3", "Miami"),
        ("B3", "Orlando"),
    ],
    StructType(
        [
            StructField("airport_id", StringType(), True),
            StructField("airport_name", StringType(), True),
        ]
    ),
)

# Create planes dataframe
planes = spark.createDataFrame(
    [(1, "Airbus A320"), (2, "Boeing 737"), (3, "Airbus A380")],
    StructType(
        [
            StructField("plane_id", IntegerType(), True),
            StructField("plane_model", StringType(), True),
        ]
    ),
)

In [2]:
flights_with_airports = flights.join(
    airports.withColumnRenamed("airport_id", "origin_airport").withColumnRenamed(
        "airport_name", "origin_airport_name"
    ),
    on="origin_airport",
    how="left",
).join(
    airports.withColumnRenamed("airport_id", "destination_airport").withColumnRenamed(
        "airport_name",
        "destination_airport_name",
    ),
    on="destination_airport",
    how="left",
)

# Join the resulting dataframe with planes to get the plane models
flights_with_airports_and_planes = flights_with_airports.join(
    planes.withColumnRenamed("plane_id", "flight_id").withColumnRenamed(
        "plane_model", "plane_model_name"
    ),
    on="flight_id",
    how="left",
)

# Create a new dataframe with the required columns and their lengths
result = flights_with_airports_and_planes.select(
    col("flight_id"),
    length(col("origin_airport_name")).alias("origin_airport_name_length"),
    length(col("destination_airport_name")).alias("destination_airport_name_length"),
    length(col("plane_model_name")).alias("plane_model_length"),
)

In [3]:
result.show()

+---------+--------------------------+-------------------------------+------------------+
|flight_id|origin_airport_name_length|destination_airport_name_length|plane_model_length|
+---------+--------------------------+-------------------------------+------------------+
|        1|                        13|                             11|                11|
|        3|                         5|                              7|                11|
|        2|                         8|                              6|                10|
+---------+--------------------------+-------------------------------+------------------+

